<a href="https://colab.research.google.com/github/Markkwell/Project-YpYt/blob/main/project1_by_Marko_Deko.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Your path Your trials





In [45]:
# ИМПОРТ БИБЛИОТЕК
import re

import numpy as np
import pandas as pd

In [46]:
# ИМПОРТ ДАТАСЕТА И КОНСТАНТЫ
# -Настройка путей и проверка структуры данных
DATASET_PATH = "steam_reviews.csv"

REQUIRED_COLUMNS = (
    "app_id",
    "app_name",
    "review_text",
    "review_score",
    "review_votes"
)

In [47]:
# ФУНКЦИИ ОБРАБОТКИ ДАННЫХ
# -Загрузка, очистка и подготовка отзывов

def load_dataset(path: str) -> pd.DataFrame:
    """
    Загружает CSV и проверяет структуру.
    """

    try:
        df = pd.read_csv(path)

    except FileNotFoundError:
        raise ValueError(f"Файл не найден: {path}")

    if df.empty:
        raise ValueError("Датасет пуст.")

    missing_columns = [
        column
        for column in REQUIRED_COLUMNS
        if column not in df.columns
    ]

    if missing_columns:
        raise ValueError(
            f"Отсутствуют столбцы: {missing_columns}"
        )

    return df


def clean_text(text: str) -> str:
    """
    Очищает текст отзыва.
    """

    text = str(text).lower()
    text = re.sub(r"[^a-zA-Z ]", "", text)
    text = re.sub(r"\s+", " ", text)

    return text.strip()


def prepare_reviews(df: pd.DataFrame) -> pd.DataFrame:
    """
    Создает очищенный текст
    и определяет тип отзыва.
    """

    prepared_df = df.copy()

    prepared_df["app_name"] = prepared_df["app_name"].fillna("Unknown")
    prepared_df["review_text"] = prepared_df["review_text"].fillna("")

    prepared_df["clean_review"] = [
        clean_text(text)
        for text in prepared_df["review_text"]
    ]

    prepared_df["sentiment"] = np.where(
        prepared_df["review_score"] > 0,
        "Positive",
        "Negative"
    )

    return prepared_df

In [48]:
# ФУНКЦИИ АНАЛИТИКИ
# -Анализ отзывов и вычисление статистики игр

def get_game_rating(df: pd.DataFrame) -> pd.DataFrame:
    """
    Создает статистику отзывов по играм.
    """

    grouped = (
        df.groupby(["app_name", "sentiment"])
        .size()
        .unstack(fill_value=0)
    )

    grouped["Positive"] = grouped.get("Positive", 0)
    grouped["Negative"] = grouped.get("Negative", 0)

    grouped["ratio"] = (
        grouped["Positive"] /
        (grouped["Negative"] + 1)
    )

    grouped["total_reviews"] = (
        grouped["Positive"] +
        grouped["Negative"]
    )

    return grouped.reset_index()


def get_reviews(
    df: pd.DataFrame,
    sentiment: str
) -> pd.DataFrame:
    """
    Возвращает отзывы выбранного типа.
    """

    return df[
        df["sentiment"] == sentiment
    ][
        ["app_name", "review_text"]
    ].head(5)


def top_r_games(df: pd.DataFrame) -> pd.DataFrame:
    """
    Возвращает 5 игр с лучшим P/N.
    """

    stats = get_game_rating(df)

    return stats.sort_values(
        by="ratio",
        ascending=False
    ).head(5)


def lead_comm(df: pd.DataFrame) -> pd.DataFrame:
    """
    Возвращает 5 игр
    с наибольшим числом отзывов.
    """

    stats = get_game_rating(df)

    return stats.sort_values(
        by="total_reviews",
        ascending=False
    ).head(5)


def random_comm(
    df: pd.DataFrame,
    game_name: str,
    review_count: int
) -> pd.DataFrame:
    """
    Возвращает случайные отзывы игры.
    """

    result = df[
        df["app_name"].str.contains(
            game_name,
            case=False,
            na=False
        )
    ]

    if result.empty:
        raise ValueError("Игра не найдена.")

    sample_size = min(
        review_count,
        len(result)
    )

    return result.sample(
        sample_size,
        random_state=np.random.randint(0, 1000)
    )


def game_list(df: pd.DataFrame) -> list:
    """
    Возвращает список игр.
    """

    return sorted(
        df["app_name"]
        .dropna()
        .unique()
    )


def compare_games(
    df: pd.DataFrame,
    game_1: str,
    game_2: str
) -> pd.DataFrame:
    """
    Возвращает статистику
    для двух игр.
    """

    stats = get_game_rating(df)

    games = stats[
        stats["app_name"].isin(
            [game_1, game_2]
        )
    ]

    if len(games) < 2:
        raise ValueError(
            "Одна из игр не найдена."
        )

    return games

In [49]:
# ФУНКЦИИ ИНТЕРФЕЙСА И КОМАНД
# -Меню и обработка пользовательских команд

def print_menu() -> None:
    """
    Показывает меню.
    """

    print("\nКоманды:")
    print("Positive reviews")
    print("Negative reviews")
    print("Top R")
    print("Lead comm")
    print("Random comm <game> <n>")
    print("Game list")
    print("Compare <game1> | <game2>")
    print("0 - выход")

In [50]:
# ВЫВОД И ОБРАБОТКА КОМАНД

def handle_command(
    command: str,
    df: pd.DataFrame
) -> bool:
    """
    Обрабатывает команду пользователя.
    """

    command_lower = command.lower()

    try:

        if command_lower == "0":
            print("Работа завершена.")
            return False

        if command_lower == "positive reviews":

            reviews = get_reviews(
                df,
                "Positive"
            )

            print("\nPositive reviews:\n")

            for _, row in reviews.iterrows():
                print(f"Game: {row['app_name']}")
                print(f"Review: {row['review_text']}")
                print("-" * 40)

            return True

        if command_lower == "negative reviews":

            reviews = get_reviews(
                df,
                "Negative"
            )

            print("\nNegative reviews:\n")

            for _, row in reviews.iterrows():
                print(f"Game: {row['app_name']}")
                print(f"Review: {row['review_text']}")
                print("-" * 40)

            return True

        if command_lower == "top r":

            result = top_r_games(df)

            print("\nTop R:\n")

            for _, row in result.iterrows():
                print(
                    f"{row['app_name']} | "
                    f"P={row['Positive']} | "
                    f"N={row['Negative']} | "
                    f"P/N={row['ratio']:.2f}"
                )

            return True

        if command_lower == "lead comm":

            result = lead_comm(df)

            print("\nLead comm:\n")

            for _, row in result.iterrows():
                print(
                    f"{row['app_name']} | "
                    f"Reviews="
                    f"{row['total_reviews']}"
                )

            return True

        if command_lower == "game list":

            games = game_list(df)

            print("\nGame list:\n")

            for game in games:
                print(game)

            return True

        if command_lower.startswith(
            "random comm "
        ):

            raw = command[len("Random comm "):]
            last_space = raw.rfind(" ")

            game = raw[:last_space]
            count = int(raw[last_space + 1:])

            result = random_comm(
                df,
                game,
                count
            )

            print("\nRandom comments:\n")

            for _, row in result.iterrows():
                print(
                    f"[{row['sentiment']}] "
                    f"{row['review_text']}"
                )
                print("-" * 40)

            return True

        if command_lower.startswith(
            "compare "
        ):

            raw = command[len("Compare "):]

            game_1, game_2 = [
                item.strip()
                for item in raw.split("|", 1)
            ]

            result = compare_games(
                df,
                game_1,
                game_2
            )

            print("\nCompare:\n")

            for _, row in result.iterrows():
                print(
                    f"{row['app_name']} | "
                    f"P={row['Positive']} | "
                    f"N={row['Negative']} | "
                    f"P/N={row['ratio']:.2f}"
                )

            return True

        print("Неизвестная команда.")

    except Exception as error:
        print(f"Ошибка: {error}")

    return True

In [52]:
# ВЫВОД И ЗАПУСК ПРОГРАММЫ
# -Главная функция и запуск системы

def main() -> None:
    """
    Запуск системы.
    """

    try:

        df = load_dataset(DATASET_PATH)
        prepared_df = prepare_reviews(df)

        print(
            f"\nЗагружено отзывов: "
            f"{len(prepared_df)}"
        )

        while True:

            print_menu()

            command = input(
                "\nВведите команду: "
            )

            if not handle_command(
                command,
                prepared_df
            ):
                break

    except Exception as error:
        print(f"Ошибка: {error}")


if __name__ == "__main__":
    main()


Загружено отзывов: 49999

Команды:
Positive reviews
Negative reviews
Top R
Lead comm
Random comm <game> <n>
Game list
Compare <game1> | <game2>
0 - выход

Введите команду: 0
Работа завершена.
